---
title: "UN Comtrade: International Trade Data"
subtitle: "NoSQL Lab Personal Project — Extraction Pilot"
lang: en
format:
  pdf:
    documentclass: scrartcl
    papersize: a4
    fontsize: 11pt
    geometry:
      - margin=25mm
    toc: true
    toc-depth: 2
    number-sections: true
    colorlinks: true
    code-overflow: wrap
execute:
  enabled: false
  echo: true
---

# Summary and current scope

This notebook starts with an **extraction pilot** for the NoSQL personal project. It tests authenticated access and inspects the returned records before choosing the final research scope.

The pilot requests annual US exports of HS chapter **85** (electrical machinery and equipment) to the world for **2022 and 2023**. These are test settings, not the final research topic. Two aggregate observations are not enough for the final in-depth analysis.

**Current workflow:** configure → request JSON → collect raw records → inspect and validate.

**Later:** load into MongoDB → clean and model inside MongoDB → analyse with aggregation pipelines → add diagrams and export the report. No database writes occur in this version.

# Requirements and configuration

Use a Python kernel with `requests`, `pandas`, and `python-dotenv` installed. Install the repository-local dependencies with `python -m pip install -r requirements.txt` from this notebook's folder, using the same Python environment as the notebook kernel. `python-dotenv` loads a local `.env` file; the other imports belong to Python's standard library.

The course template uses `requests.get()`, `.json()`, a list of records, and a loop with a delay. I will keep that approach. Authentication, query parameters, and response fields follow the UN Comtrade documentation. Timeouts and bounded rate-limit retries are small reliability additions.

In [1]:
import os
import time
import json
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
from pprint import pprint

import pandas as pd
import requests
from dotenv import load_dotenv

# Keep notebook tables compact enough for reading and later PDF export.
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_colwidth", 60)

## Reproduction options

**Option A — fresh API extraction:** leave `DATA_SOURCE = "api"`. The notebook downloads data through the API and later saves a dated raw-data snapshot after validation. Source revisions can change results between downloads.

**Option B — saved screenshot data:** set `DATA_SOURCE = "snapshot"` and set `SNAPSHOT_PATH` to the JSON file supplied in the project's `data` folder. Restart the kernel and run from the top. No API key or network request is needed for this mode; it reads the same raw records and query log used for the saved results. This option supports reproducibility and does not replace the assignment's demonstrated API extraction.

The path below is a placeholder until the first snapshot is created. Use the dated filename printed by the saving cell. Both modes must match this pilot's query settings and pass the same record validation.

In [ ]:
DATA_SOURCE = "api"
SNAPSHOT_PATH = Path("data/trade_snapshot.json")

if DATA_SOURCE not in {"api", "snapshot"}:
    raise ValueError("Choose either api or snapshot as DATA_SOURCE.")

## API key

**Only needed for fresh extraction; skipped in snapshot mode.** Use your own free key:

1. Follow UN Comtrade's [free account registration instructions](https://uncomtrade.org/docs/how-to-create-an-account/).
2. Open [My API Portal](https://comtradeplus.un.org/MyComtrade/APIPortal), generate a key if needed, and copy either the primary or secondary key. Free Basic Individual access is sufficient for this pilot.
3. Set `DATA_SOURCE = "api"`. Copy `.env.example` to `.env` beside the notebook and fill in `UN_COMTRADE_API_KEY=` with your key, or use the hidden prompt.
4. Restart the kernel and run from the top with this notebook's folder as the working directory. Existing environment values take precedence over `.env`.

Reference: the [official example notebook](https://github.com/uncomtrade/comtradeapicall/blob/main/tests/example%20calling%20functions%20-%20notebook.ipynb) uses the `comtradeapicall` helper package. Here we use `requests` directly to follow the course; no helper package is required.

To reproduce without a key, choose **snapshot mode** and the supplied dated JSON file. Do not submit `.env` or show keys in code or outputs.

In [ ]:
headers = {}
# Read the secret without writing it into the notebook source.
if DATA_SOURCE == "api":
    load_dotenv(dotenv_path=".env", override=False)
    api_key = os.environ.get("UN_COMTRADE_API_KEY", "").strip()
    if not api_key:
        api_key = getpass("UN Comtrade API key: ").strip()

    if not api_key:
        raise ValueError("An API key is required for this authenticated endpoint.")

    # A header keeps the key out of the query URL.
    headers = {"Ocp-Apim-Subscription-Key": api_key}
    print("API key loaded. Its value is not displayed.")
else:
    print("Snapshot mode: no API key is loaded or requested.")

## Pilot query settings

| Setting | Meaning |
|---|---|
| `C / A / HS` in the endpoint | Goods, annual frequency, original HS classification |
| `reporterCode = 842` | United States, the country reporting the trade |
| `partnerCode = 0` | World total, not a particular destination |
| `cmdCode = "85"` | One two-digit commodity chapter; keep codes as strings |
| `flowCode = "X"` | Exports |
| `partner2Code = 0`, `customsCode = "C00"`, `motCode = 0` | Aggregate selections for the additional dimensions |
| `includeDesc = "true"` | Request readable descriptions alongside codes |
| `maxRecords = 500` | Conservative request cap, not a guarantee of completeness |

We request one year at a time. The original `HS` classification can resolve to different editions across countries or years; inspect the returned `classificationCode` before comparing them. The final project may instead need a fixed edition.

typeCode(str) : Product type. Goods (C) or Services (S)
freqCode(str) : The time interval at which observations occur. Annual (A) or Monthly (M)
clCode(str) : Indicates the product classification used and which version (HS, SITC)
period(str) : Combination of year and month (for monthly), year for (annual)
reporterCode(str) : The country or geographic area to which the measured statistical phenomenon relates
cmdCode(str) : Product code in conjunction with classification code
flowCode(str) : Trade flow or sub-flow (exports, re-exports, imports, re-imports, etc.)
partnerCode(str) : The primary partner country or geographic area for the respective trade flow
partner2Code(str) : A secondary partner country or geographic area for the respective trade flow
customsCode(str) : Customs or statistical procedure
motCode(str) : The mode of transport used when goods enter or leave the economic territory of a country

In [3]:
API_URL = "https://comtradeapi.un.org/data/v1/get/C/A/HS"
YEARS = ["2022", "2023"]
MAX_RECORDS = 500
REQUEST_DELAY = 5

# These filters describe one aggregate observation per year.
base_params = {
    "reporterCode": 842,
    "partnerCode": 0,
    "partner2Code": 0,
    "cmdCode": "85",
    "flowCode": "X",
    "customsCode": "C00",
    "motCode": 0,
    "maxRecords": MAX_RECORDS,
    "includeDesc": "true"
}

# ELT process

## Extract: request a single year

`requests.get()` downloads the response. `.raise_for_status()` stops on an HTTP error, and `.json()` converts JSON into Python dictionaries and lists. Comtrade's actual trade records are in `payload["data"]`, rather than the cocktail template's `data["drinks"]`.

The function retries HTTP **429** (rate limit) at most twice after the first attempt. Other HTTP failures stop immediately. For **401/403**, check the key and account access rather than repeatedly retrying. A response reaching the record cap is rejected until the query is narrowed or a documented larger extraction strategy is chosen.

In [4]:
def extract_year(year):
    params = base_params.copy()
    params["period"] = year

    for attempt in range(3):
        response = requests.get(
            API_URL, headers=headers, params=params, timeout=30
        )

        # Pause and retry only when the service reports a rate limit.
        if response.status_code == 429 and attempt < 2:
            wait_seconds = 15 * (attempt + 1)
            print(f"Rate limit for {year}; retrying in {wait_seconds} seconds.")
            time.sleep(wait_seconds)
            continue

        response.raise_for_status()
        payload = response.json()

        if not isinstance(payload, dict):
            raise ValueError("Expected a JSON object from Comtrade.")
        if payload.get("error"):
            raise ValueError("Comtrade reported an API error; review the query and documentation.")
        if not isinstance(payload.get("data"), list):
            raise ValueError("Expected a list of records in the API's data field.")

        records = payload["data"]
        if len(records) >= MAX_RECORDS:
            raise ValueError("The record cap was reached; completeness is not established.")

        return records

## Extract: collect the selected years

Like the course template, we loop over query values, add the returned documents to a list, and pause between requests. Here the query values are years, not the first letters of drink names.

`list_trade` preserves the returned trade fields. `extraction_log` separately records when each query was made and how many records it returned. In snapshot mode, these are restored from the supplied JSON file instead. This cell does not write to MongoDB or save a new file. Snapshot query metadata must match the current settings.

In [ ]:
list_trade = []
extraction_log = []
query_scope = {"endpoint": API_URL, "years": YEARS, "filters": base_params}

if DATA_SOURCE == "snapshot":
    with SNAPSHOT_PATH.open(encoding="utf-8") as file:
        snapshot = json.load(file)
    if not isinstance(snapshot, dict) or snapshot.get("schema_version") != 1:
        raise ValueError("Unsupported snapshot format.")
    if snapshot.get("query_scope") != query_scope:
        raise ValueError("Snapshot query settings differ from this notebook's settings.")
    if not isinstance(snapshot.get("records"), list):
        raise ValueError("Snapshot must contain a list of raw records.")
    if not isinstance(snapshot.get("extraction_log"), list):
        raise ValueError("Snapshot must contain an extraction log.")
    list_trade = snapshot["records"]
    extraction_log = snapshot["extraction_log"]
    print(f"Loaded {len(list_trade)} raw records from {SNAPSHOT_PATH}.")
else:
    for i, year in enumerate(YEARS):
        print(f"Downloading {year} ({i + 1}/{len(YEARS)})")
        extracted_at = datetime.now(timezone.utc).isoformat()
        records = extract_year(year)

        # This exact pilot query should return one aggregate record per year.
        if len(records) != 1:
            raise ValueError(
                f"Expected one record for {year}, received {len(records)}. "
                "Check availability and query scope before continuing."
            )

        list_trade.extend(records)
        extraction_log.append({
            "year_requested": year,
            "records_received": len(records),
            "extracted_at_utc": extracted_at,
            "endpoint": API_URL,
            "query": {**base_params, "period": year}
        })

        if i < len(YEARS) - 1:
            time.sleep(REQUEST_DELAY)

    print(f"Extraction finished: {len(list_trade)} raw records.")

## Inspect and validate the raw extraction

First inspect one full record, as in the course template. Then check that the returned dimensions match the requested pilot. These are extraction checks, not the final MongoDB cleaning step.

We intentionally keep missing measurements and quality flags unchanged. A zero quantity with an unavailable unit is not automatically a meaningful measured zero. World and commodity totals must not later be summed together with their component records.

In [6]:
# Inspect a raw trade document, not the secret-bearing request headers.
pprint(list_trade[0])

{'aggrLevel': 2,
 'altQty': 0.0,
 'altQtyUnitAbbr': 'N/A',
 'altQtyUnitCode': -1,
 'cifvalue': None,
 'classificationCode': 'H6',
 'classificationSearchCode': 'HS',
 'cmdCode': '85',
 'cmdDesc': 'Electrical machinery and equipment and parts thereof; sound '
            'recorders and reproducers; television image and sound recorders '
            'and reproducers, parts and accessories of such articles',
 'customsCode': 'C00',
 'customsDesc': 'TOTAL CPC',
 'flowCode': 'X',
 'flowDesc': 'Export',
 'fobvalue': 197748996567.0,
 'freqCode': 'A',
 'grossWgt': 0.0,
 'isAggregate': True,
 'isAltQtyEstimated': False,
 'isGrossWgtEstimated': False,
 'isLeaf': False,
 'isNetWgtEstimated': False,
 'isOriginalClassification': True,
 'isQtyEstimated': False,
 'isReported': False,
 'legacyEstimationFlag': 0,
 'mosCode': '0',
 'motCode': 0,
 'motDesc': 'TOTAL MOT',
 'netWgt': None,
 'partner2Code': 0,
 'partner2Desc': 'World',
 'partner2ISO': 'W00',
 'partnerCode': 0,
 'partnerDesc': 'World',
 'partn

In [7]:
if len(list_trade) != len(YEARS):
    raise ValueError("Incomplete or unexpected pilot record count; rerun extraction successfully.")

returned_periods = [str(record.get("period")) for record in list_trade]
if sorted(returned_periods) != sorted(YEARS):
    raise ValueError("Returned periods do not match the complete requested pilot.")

expected_fields = {
    "reporterCode": base_params["reporterCode"],
    "partnerCode": base_params["partnerCode"],
    "partner2Code": base_params["partner2Code"],
    "cmdCode": base_params["cmdCode"],
    "flowCode": base_params["flowCode"],
    "customsCode": base_params["customsCode"],
    "motCode": base_params["motCode"]
}

for record, year in zip(list_trade, YEARS):
    if str(record.get("period")) != year:
        raise ValueError("A returned period does not match the request.")
    for field, expected in expected_fields.items():
        if record.get(field) != expected:
            raise ValueError(f"Unexpected value for {field}; review the response.")
    if record.get("primaryValue") is None:
        raise ValueError("A requested record has no primary trade value.")
    if not record.get("classificationCode"):
        raise ValueError("A returned record has no classification edition.")

editions = {record["classificationCode"] for record in list_trade}
if len(editions) != 1:
    raise ValueError("Multiple HS editions returned; resolve comparability before analysis.")

print("Pilot dimensions and trade-value presence checked.")
print("Returned HS edition:", sorted(editions))

Pilot dimensions and trade-value presence checked.
Returned HS edition: ['H6']


## Save the validated raw-data snapshot

In API mode, this cell saves raw records, the query scope, and original download timestamps together in a dated JSON file. It uses a new filename and exclusive creation rather than overwriting an existing snapshot. It never stores API headers, keys, or database credentials. Snapshot mode leaves the supplied file unchanged.

The professor can use this file with Option B above to repeat the later processing on the same input data. Include the selected snapshot in the submission only after checking ILIAS/course upload guidance and the source's terms; keep it separate from secret files.

UN Comtrade is not an unrestricted redistribution license. Its [re-dissemination guidance](https://uncomtrade.org/docs/re-dissemination-of-data/) describes exceptions for limited publications, internal use, and specified free analytical/data-sharing uses. Cite UN Comtrade and check the applicable terms before distributing a larger dataset or publishing raw snapshots on GitHub.

In [ ]:
if DATA_SOURCE == "api":
    if len(list_trade) != len(YEARS) or len(extraction_log) != len(YEARS):
        raise ValueError("Complete extraction and validation before saving the pilot.")

    saved_at = datetime.now(timezone.utc)
    snapshot_data = {
        "schema_version": 1,
        "source": "UN Comtrade",
        "source_url": "https://comtradeplus.un.org/ApiDocs",
        "saved_at_utc": saved_at.isoformat(),
        "query_scope": query_scope,
        "extraction_log": extraction_log,
        "records": list_trade
    }
    snapshot_folder = Path("data")
    snapshot_folder.mkdir(exist_ok=True)
    snapshot_name = f"trade_snapshot_{saved_at:%Y%m%dT%H%M%S%fZ}.json"
    saved_snapshot_path = snapshot_folder / snapshot_name

    with saved_snapshot_path.open("x", encoding="utf-8") as file:
        json.dump(snapshot_data, file, ensure_ascii=False, indent=2, allow_nan=False)

    print(f"Saved {len(list_trade)} raw records to {saved_snapshot_path}.")
    print("For offline reproduction, select snapshot mode and use this path.")
else:
    print("Using the supplied snapshot; no new snapshot was written.")

In [8]:
# Pandas is only for inspection here; list_trade remains the raw input for MongoDB.
preview_columns = [
    "period", "reporterDesc", "partnerDesc", "cmdCode",
    "classificationCode", "primaryValue", "netWgt", "isNetWgtEstimated"
]
df_preview = pd.DataFrame(list_trade)
df_preview.reindex(columns=preview_columns)

,period,reporterDesc,partnerDesc,cmdCode,classificationCode,primaryValue,netWgt,isNetWgtEstimated
0,2022,USA,World,85,H6,1.977490e+11,NaN,False
1,2023,USA,World,85,H6,2.006547e+11,0.0,True


In [9]:
# The log contains query settings and timestamps, but no API key.
pd.DataFrame(extraction_log)[
    ["year_requested", "records_received", "extracted_at_utc"]
]

,year_requested,records_received,extracted_at_utc
0,2022,1,2026-10-15T17:28:40.235313+00:00
1,2023,1,2026-10-15T17:28:46.444481+00:00


## Pilot interpretation and next steps

A successful run should produce two raw observations, one per selected year, with matching query dimensions and one returned HS edition. This validates a narrow access path; it does not establish completeness for a larger query or provide the final project's conclusions.

Before expanding, choose the reporting country, research questions, commodity level, time range, and partner scope. Revisit the one-record-per-year check if those settings change. Keep a single commodity level and separate World totals from partner breakdowns to avoid double-counting.

Next, load an agreed raw dataset into MongoDB with a repeatable import strategy. Perform the main cleaning, embedded-document modeling, and analytical aggregations inside MongoDB. The final submission still needs the class diagram, a second image, conclusions, learnings, and a checked PDF.

# Sources and method references

- [UN Comtrade API documentation](https://comtradeplus.un.org/ApiDocs): endpoints, query parameters, and authentication.
- [UN Comtrade API guide](https://uncomtrade.org/docs/un-comtrade-api/): public preview versus authenticated access.
- [Subscription plans](https://uncomtrade.org/docs/subscriptions/): check current free-tier limits before a larger extraction.
- [Commodity classification and conversion issues](https://uncomtrade.org/docs/reported-and-converted-data-commodity-conversion-issues/): HS revisions and comparability.
- [Trade valuation](https://uncomtrade.org/docs/trade-valuation/): interpretation of primary trade values and import/export valuation differences.
- [Course project template](https://wiki.revolytics.com/38c8b4dc7e3246018682b651ceefbd95/): the `requests` → JSON → list → load workflow.
- [Local course notes](../txt%20course%20content/Copy_paste_course_content.md): PyMongo insertion, document modeling, and MongoDB transformations for subsequent stages.

Trade values are in current US dollars. They are not inflation-adjusted. Download timestamps are recorded when the extraction cell runs; source data may be revised.

# PDF export and final submission checklist

The first Markdown cell contains Quarto YAML settings. Quarto supplies the title, table of contents, and section numbering, so headings do not contain manually typed numbers. Code remains visible to document the method. Rendering is configured **not to execute Python cells**: the PDF uses the notebook's saved outputs.

From a terminal opened in this notebook's folder, render with:

```bash
quarto render "UN_commodity.ipynb" --to pdf --no-execute
```

Before the final export:

- Replace the pilot subtitle with the final project subtitle and add your author details to the YAML header.
- Complete the analysis, conclusions, learnings, and source references; remove or rewrite this working checklist for the submitted report.
- Include at least two images, one being the class diagram. Use relative paths, readable labels, and captions.
- Keep result tables narrow, with clear units and sensible precision. A Pandas table that looks good in Jupyter may need adjustment for LaTeX PDF output.
- Restart the kernel, run all cells successfully in Jupyter, and save. Running the notebook performs the configured requests and, once implemented, database operations; inspect the rerun policy first.
- Check code and saved outputs for credentials, tracebacks, excessive logs, and unnecessary raw dumps. Hide verbose cells selectively for the PDF without removing reproducible code from the notebook.
- Render and inspect every page for clipped code or tables, broken images, duplicated numbering, and awkward page breaks. This pilot PDF has no data results until the extraction cells have actually run and their outputs have been saved.
- Package the final PDF, executed notebook, and image files in a ZIP. If the upload rules accept additional files, include the selected dated snapshot, `.env.example`, dependencies, and rerun instructions. Exclude `.env`, unused snapshots, and temporary render artifacts. Check relative image and snapshot paths after extracting the ZIP.

Quarto and TinyTeX were available during the initial setup check. If rendering later becomes impractical, browser Print → Save as PDF is an assignment-approved fallback.